# 08 — Generate paper figures other than Figure 1

Read finalized products and generate publication figures. Scientific measurements are loaded from CSV rather than silently recalculated.

08_generate_paper_figures.ipynb
Useful in concept, but currently incomplete and too early in numbering.
It only covers:
* initial second-stage failure;
* principal explosion;
* capsule sequence;
* some array-result figures.
It does not yet appear to generate the complete planned figure set, and it predates much of the newer audio/reduced-time figure work.
Rename:
12_generate_paper_figures.ipynb
Refactor it so that it:
* contains almost no scientific processing;
* reads frozen CSV/JSON/miniSEED outputs;
* uses shared plotting functions;
* generates all final main and supplementary figures;
* records figure input files and settings;
* produces deterministic PNG and PDF outputs.
This is where the newer synchronized audio/reduced-time plot should eventually live.
Verdict: keep but rewrite as a pure figure-production notebook.

In [ ]:

from pathlib import Path
import sys

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
MODULE_DIR = PROJECT_ROOT / "modules"
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from obspy import Stream, UTCDateTime

from project_config import ensure_output_dirs

PATHS = ensure_output_dirs(PROJECT_ROOT)
DERIVED_DIR = PATHS["derived"]
FIGURE_DIR = PATHS["figures"]

plt.rcParams.update({
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "figure.dpi": 120,
})

print(DERIVED_DIR)


In [ ]:

from obspy import read
from plotting import plot_key_event_waveforms

st_corr = read(str(DERIVED_DIR / "bchh_corrected_analysis_window.pkl"),
               format="PICKLE")
measurements = pd.read_csv(
    DERIVED_DIR / "key_event_pressure_measurements.csv"
)


## Initial second-stage failure

In [ ]:

EXPLOSION_TIME = UTCDateTime("2016-09-01T13:07:12.080")
t0, t1 = EXPLOSION_TIME + 3.0 - 0.08, EXPLOSION_TIME + 5.0 - 0.08
st_event = st_corr.copy().trim(t0, t1)
event_results = measurements.loc[
    measurements["event"] == "Initial second-stage failure"
]
plot_key_event_waveforms(
    st_event,
    reference_time=t0,
    pressure_results=event_results,
    title="Initial second-stage failure",
    outfile=FIGURE_DIR / "second_stage_waveforms.png",
)
plt.show()


## Principal explosion

In [ ]:

t0, t1 = EXPLOSION_TIME + 6.0 - 0.08, EXPLOSION_TIME + 9.0 - 0.08
st_event = st_corr.copy().trim(t0, t1)
event_results = measurements.loc[
    measurements["event"] == "Principal explosion"
]
plot_key_event_waveforms(
    st_event,
    reference_time=t0,
    pressure_results=event_results,
    title="Principal Falcon 9 explosion",
    outfile=FIGURE_DIR / "principal_explosion_waveforms.png",
)
plt.show()


## Capsule sequence

In [ ]:

t0 = UTCDateTime("2016-09-01T13:07:27.0")
t1 = UTCDateTime("2016-09-01T13:07:30.0")
st_event = st_corr.copy().trim(t0, t1)
capsule_results = measurements.loc[
    measurements["event"].isin(["Capsule pulse 1", "Capsule pulse 2"])
]
# For a two-pulse figure, marker annotations should be added manually or by
# extending plotting.py to accept multiple measurement windows.
plot_key_event_waveforms(
    st_event,
    reference_time=t0,
    pressure_results=None,
    title="Capsule-related acoustic pulses",
    outfile=FIGURE_DIR / "capsule_waveforms.png",
    add_measurements=False,
)
plt.show()


## Array-result figures

In [ ]:

array_file = DERIVED_DIR / "event_catalogue_array_results.csv"
if array_file.exists():
    array_results = pd.read_csv(array_file)

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.scatter(
        np.arange(len(array_results)),
        array_results["back_azimuth_deg"],
        c=array_results["mean_abs_correlation"],
        s=22,
    )
    ax.set_xlabel("Event number")
    ax.set_ylabel("Back azimuth (degrees)")
    ax.set_title("Source direction through the explosion sequence")
    fig.savefig(
        FIGURE_DIR / "catalogue_back_azimuth.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()
else:
    print("Run 04_array_analysis.ipynb on the full catalogue first.")
